# 01 - Train and evaluate

Run **00_build_library.ipynb first**. This notebook imports the modules,
builds the BreakHis datasets (patient-level split), trains the proposed
PPG-SwinT model and the three baselines, and shows a metrics table.

**Set `DATA_ROOT` below** to the folder that contains the BreakHis PNGs.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('ppg_breakhis'))
import torch
from config import Config
from data.breakhis import make_datasets
from models.backbone import SwinFeatureExtractor
from models.baselines import build_model
from engine.trainer import train
from train import set_seed

## 1. Configure

In [3]:
DATA_ROOT = '/content/BreakHis_v1'   # <-- EDIT THIS

cfg = Config()
cfg.data_root = DATA_ROOT
cfg.magnification = '200X'
cfg.epochs = 30            # lower to e.g. 5 for a fast first look
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(cfg.seed)
print('device:', cfg.device)

device: cpu


## 2. Build datasets (patient-level split)
The split is by patient, not by image - the loader asserts no patient
leaks between train/val/test.

In [5]:
datasets = make_datasets(cfg)
print({k: len(v) for k, v in datasets.items()})

RuntimeError: No 200X images found under /content/BreakHis_v1. Check data_root and that filenames follow the SOB_* convention.

## 3. Train the proposed model
`pretrained=True` downloads Swin-Tiny weights on first run (needs
internet). Set `cfg.pretrained = False` to skip.

In [ ]:
cfg.exp_name = 'ppg_swint'
backbone = SwinFeatureExtractor(cfg.backbone, pretrained=cfg.pretrained,
                                stage=cfg.feature_stage)
model = build_model(cfg, backbone)
model, ppg_metrics = train(model, datasets, cfg)
ppg_metrics

## 4. Train the baselines
Each isolates one claim: `plain` (is any machinery needed?), `proto`
(does the probabilistic part beat plain prototypes?), `mcdropout`
(does the prototype part add anything over uncertainty?).

In [ ]:
import copy
results = {'ppg_swint': ppg_metrics}
for exp in ['plain', 'proto', 'mcdropout']:
    c = copy.deepcopy(cfg); c.exp_name = exp
    set_seed(c.seed)
    bb = SwinFeatureExtractor(c.backbone, pretrained=c.pretrained, stage=c.feature_stage)
    _, results[exp] = train(build_model(c, bb), datasets, c)
results

## 5. Metrics table

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T[['accuracy', 'auc', 'f1', 'ece', 'nll']]
df = df.rename_axis('model').round(4)
df